# Machine Learning: Supervised Learning after Anomaly Removal
**Copyright © 2026 Sujeet Banerjee**  
**Email:** sujee.banerjee@gmail.com

---

Welcome to this advanced beginner workbook! In this session, we will combine Supervised and Unsupervised learning:
1. We will train a **Baseline Linear Regression** model on the raw California housing data.
2. We will use an **Unsupervised Model (One-Class SVM)** to find and remove anomalies (outliers) from our training data.
3. We will train a **New Linear Regression** model on this newly cleaned dataset.

**The Goal:** To see if removing the weirdest 5% of our training data helps the Linear Regression model make better, more accurate predictions on the test set (measured by a lower RMSE).

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

# 1. Load the data
print("1. Loading California Housing Data...")
housing = fetch_california_housing(as_frame=True)
X = housing.data
Y = housing.target

# 2. Split into Training and Testing sets FIRST
# RULE: We only remove outliers from the TRAINING set. 
# The test set must represent the real world, which includes weird data!
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
print(f"Original Training Set Size (outliers included): {X_train.shape[0]} data-points")
print(f"Original Test Set Size (outliers included): {X_test.shape[0]} data-points")

1. Loading California Housing Data...
Original Training Set Size (outliers included): 16512 data-points
Original Test Set Size (outliers included): 4128 data-points


### Step 3: Train the Baseline Model (With Outliers)
First, let's train a model on the raw, unfiltered data so we have a score to beat.

In [5]:
# Train baseline model
baseline_model = LinearRegression()
baseline_model.fit(X_train, Y_train)

# Predict and evaluate on the test set
baseline_predictions = baseline_model.predict(X_test)
baseline_rmse = np.sqrt(mean_squared_error(Y_test, baseline_predictions))

print(f"BASELINE RMSE (Trained on ALL data): {baseline_rmse:.4f}")

BASELINE RMSE (Trained on ALL data): 0.7456


### Step 4: Remove Anomalies using One-Class SVM
Now we use our unsupervised learning technique to find the 5% of training data that looks the most unusual.

In [8]:
print("Scaling data and finding anomalies in the training set...\n")

# Scale the features (Required for SVM)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Initialize and fit the One-Class SVM (nu=0.05 asks for ~5% outliers)
svm = OneClassSVM(kernel='rbf', gamma='scale', nu=0.05)
outlier_labels = svm.fit_predict(X_scaled)
print(outlier_labels)

# Filter the training data
# Keep only the rows where the SVM predicted '1' (Normal)
mask = (outlier_labels == 1)
X_clean = X[mask]
Y_clean = Y[mask]

print(f"Removed {(~mask).sum()} anomalous data points.")
print(f"Cleaned Entire Set Size: {X_clean.shape[0]} data-points")

# Split the test and train again!
X_train_clean, X_test_clean, Y_train_clean, Y_test_clean = train_test_split(X_clean, Y_clean, test_size=0.2, random_state=42)
print(f"Cleaned Training Set Size (outliers removed): {X_train_clean.shape[0]} data-points")
print(f"Cleaned Test Set Size (outliers removed): {X_test_clean.shape[0]} data-points")

Scaling data and finding anomalies in the training set...

[ 1  1 -1 ...  1  1  1]
Removed 1034 anomalous data points.
Cleaned Entire Set Size: 19606 data-points
Cleaned Training Set Size (outliers removed): 15684 data-points
Cleaned Test Set Size (outliers removed): 3922 data-points


### Step 5: Train the New Model on Clean Data
Let's see if training on this smaller, but "cleaner" dataset improves our predictions on the test set.

In [12]:
# Train new model on CLEAN data
clean_model = LinearRegression()
clean_model.fit(X_train_clean, Y_train_clean)

# Predict and evaluate on the SAME test set as before
clean_predictions = clean_model.predict(X_test_clean)
clean_rmse = np.sqrt(mean_squared_error(Y_test_clean, clean_predictions))
print(clean_model.rank_)

print("--- RESULTS ---")
print(f"BASELINE RMSE: {baseline_rmse:.4f}")
print(f"CLEANED RMSE:  {clean_rmse:.4f}")

# Assuming your trained model is named 'clean_model'
# and your test data is X_test_clean and y_test_clean

r2_score_baseline = baseline_model.score(X_test, Y_test)
print(f"R-squared Baseline: {r2_score_baseline:.4f}")

r2_score = clean_model.score(X_test_clean, Y_test_clean)
print(f"R-squared cleaned-up: {r2_score:.4f}")

if clean_rmse < baseline_rmse:
    print(f"\nSuccess! Removing anomalies reduced our error by {baseline_rmse - clean_rmse:.4f}")
else:
    print(f"\nInteresting! Removing anomalies actually increased our error by {clean_rmse - baseline_rmse:.4f}")

8
--- RESULTS ---
BASELINE RMSE: 0.7456
CLEANED RMSE:  0.6642
R-squared Baseline: 0.5758
R-squared cleaned-up: 0.6448

Success! Removing anomalies reduced our error by 0.0814


---

## 🎓 Final Keynote: The Golden Rule of the Test Set
**Why is cleaning the Test Set considered "bad" in the enterprise?**

In this workbook, we cleaned the *entire* dataset to demonstrate how mathematically sensitive Linear Regression is to outliers. You saw the RMSE drop beautifully! However, in a real-world enterprise scenario, doing this before a train/test split is a major pitfall known as **Data Leakage**.

Here is why professional Data Scientists **never** clean or filter the test set:

* **The Test Set is Reality:** The test set does not exist to make your model look good. It exists to simulate the unforgiving, messy real world. When your model is deployed to a live website, users *will* make typos, systems *will* glitch, and highly unusual data *will* occur.
* **A False Sense of Security:** If you remove the anomalies from your test set, your error rate (RMSE) will look amazing. But it is a dangerous illusion. You have tested your model in a sterilized, artificial environment, leaving you totally blind to how it will actually fail in production.
* **The Final Exam Analogy:** Imagine a teacher who wants their class to get the highest test scores in the state. Instead of teaching the students better (improving the model), the teacher just removes the 5 hardest questions from the final exam. The class average skyrockets! But did the students actually learn more? No. The test just got easier.

> **💡 The Enterprise Standard:**
> You have total freedom to manipulate your **Training Set**. You can use SVMs to drop outliers, duplicate rows, or synthesize fake data to help your model learn. But the **Test Set** must remain a raw, untouched snapshot of reality.

If your model hallucinates when it sees an outlier in the test set, that is *valuable information*! It tells the engineering team that they need to build a safety filter or use a more robust algorithm before going live.

**Remember: Train on the ideal, Test on the real.**